# AP03 Spick – In a nutshell

- **Prinzip:** `# Comment describing the action` und direkt darunter **ein Python-Einzeiler**. Die zentralen Befehle des Mock Exams sind mit `[MOCK]` markiert
- **ANPASSEN:** `df`, Dateiname, Spalten, Variablen, Features, Zielvariable, Modelltyp und Hyperparameter an deine Aufgabe anpassen. Die Beispiele bei `df` beziehen sich häufig auf die Velodaten des Mock Exams
- **WICHTIG:** Die grosse Codezelle ist **eine Befehlsbibliothek, kein zusammenhängendes Skript**. Nur benötigte Zeilen bzw. kleine Blöcke kopieren und in sinnvoller Reihenfolge ausführen. Einzeiler können Vorbedingungen haben (z. B. trainierte Modelle, vorhandene Arrays)
- **Train/Test:** Sind `X_train`, `X_test`, `y_train`, `y_test` in der Prüfung bereits vorgegeben, **nicht erneut splitten**. Trainieren/CV nur mit Training, abschliessende Bewertung mit **Test**
- **Importe:** Im Setup nach Bedarf übernehmen, in einem bereitgestellten Mock-Exam-Notebook nicht unnötig wiederholen
- **Statsmodels:** `sm.OLS()` braucht für einen Achsenabschnitt `sm.add_constant(X)`; `smf.ols("y ~ x", data=...)` enthält den Intercept standardmässig
- **Handformeln:** Bei Precision, Recall, MLE, Gini usw. dürfen die Nenner nicht 0 sein, bei fehlenden Klassen kann AUC nicht berechnet werden
- **Scikit-learn:** `predict_proba(... )[:, 1]` ist für die **positive Klasse 1** gedacht, Klassenreihenfolge mit `.classes_` prüfen
- **Threshold-Auswahl:** Einen Schwellenwert **nicht anhand der Testdaten optimieren**. Bei der Wahl zwischen Schwellenwerten ein Validierungsset innerhalb der Trainingsdaten benutzen; Testdaten erst für die abschliessende Bewertung heranziehen


## 00. Setup, Data, Features

In [ ]:
# ===== 00 - SETUP / DATA / FEATURES =====

# Import NumPy
import numpy as np

# Import pandas
import pandas as pd

# Import Matplotlib
import matplotlib.pyplot as plt

# Import statsmodels API
import statsmodels.api as sm

# Import statsmodels formula API
import statsmodels.formula.api as smf

# Import train/test split and cross-validation
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold, GridSearchCV

# Import regression models
from sklearn.linear_model import LinearRegression, LogisticRegression

# Import regression and classification trees
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree, export_text

# Import ensemble models
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier, GradientBoostingClassifier

# Import regression metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Import classification metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, RocCurveDisplay, classification_report

# Import permutation importance
from sklearn.inspection import permutation_importance

# [MOCK] Load a CSV file
df = pd.read_csv("bicycle_data.csv")

# [MOCK] Show the number of rows and columns
df.shape

# [MOCK] Display the first rows
df.head()

# Inspect column types and missing values
df.info()

# Count missing values in relevant columns
df[["age_years", "price_chf"]].isna().sum()

# Keep rows with complete model inputs and target
df_model = df.dropna(subset=["age_years", "mileage_km", "equipment_level", "is_ebike", "price_chf"]).copy()

# Remove duplicate observations if required
df_model = df_model.drop_duplicates()

# [MOCK] Select regression features (exclude the target!)
X = df[["age_years", "mileage_km", "equipment_level", "is_ebike"]]

# [MOCK] Select the numerical regression target
y = df["price_chf"]

# [MOCK] Select classification features (exclude the target!)
X_class = df[["brake_wear_pct", "chain_wear_pct", "months_since_service"]]

# [MOCK] Select the binary classification target
y_class = df["service_needed"]

# Encode a binary categorical variable (Titanic example)
df["male"] = (df["Sex"] == "male").astype(int)

# Create one-hot encoded numerical features
X = pd.get_dummies(df[["Sex", "Age", "Fare"]], columns=["Sex"], dtype=int)

# Filter records by a required time period
df_model = df[df["year"] >= 2000].copy()

## 01. Train, Test Split and Leakage Prevention

In [ ]:
# ===== 01 - TRAIN / TEST SPLIT AND LEAKAGE PREVENTION =====

# [MOCK] Split regression data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# [MOCK] Split classification data with stratification
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(X_class, y_class, test_size=0.20, random_state=42, stratify=y_class)

# [MOCK] Check regression sample sizes
print(len(X_train), len(X_test), len(y_train), len(y_test))

# Check classification class distributions
print(y_train_class.value_counts(normalize=True), y_test_class.value_counts(normalize=True))

# [MOCK] Check that no regression target is used as a feature
assert y.name not in X.columns

# Check that no classification target is used as a feature
assert y_class.name not in X_class.columns

# Check that train/test indices do not overlap
assert set(X_train.index).isdisjoint(set(X_test.index))

# Check for missing values in training features
X_train.isna().sum()


## 02. Regression Metrics, Losses & Cross-Validation

In [ ]:
# ===== 02 - REGRESSION METRICS, LOSSES AND CROSS-VALIDATION =====

# [MOCK] Predict numerical test targets
y_pred = reg_model.predict(X_test)

# Calculate prediction errors (actual minus predicted)
errors = np.asarray(y_test) - np.asarray(y_pred)

# Calculate MSE manually
mse = np.mean((np.asarray(y_test) - np.asarray(y_pred)) ** 2)

# Calculate RMSE manually
rmse = np.sqrt(np.mean((np.asarray(y_test) - np.asarray(y_pred)) ** 2))

# Calculate MSE with scikit-learn
mse = mean_squared_error(y_test, y_pred)

# [MOCK] Calculate RMSE with scikit-learn
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# Calculate MAE
mae = mean_absolute_error(y_test, y_pred)

# [MOCK] Calculate test R-squared (R²)
test_r2 = r2_score(y_test, y_pred)

# [MOCK] Round and print test R-squared
print("R-squared (test):", round(r2_score(y_test, y_pred), 3))

# Show train R-squared
print("Train R²:", reg_model.score(X_train, y_train))

# Show test R-squared
print("Test R²:", reg_model.score(X_test, y_test))

# Create a simple linear regression model
reg_model = LinearRegression()

# Fit a linear regression model
reg_model.fit(X_train, y_train)

# Get linear regression intercept
reg_model.intercept_

# Get linear regression coefficients
pd.Series(reg_model.coef_, index=X_train.columns)

# Get five-fold cross-validation R² scores
cv_scores = cross_val_score(LinearRegression(), X_train, y_train, cv=5, scoring="r2")

# Get five-fold cross-validation MSE scores
cv_mse_scores = cross_val_score(LinearRegression(), X_train, y_train, cv=5, scoring="neg_mean_squared_error")

# Convert negative CV-MSE to RMSE for each fold
cv_rmse = np.sqrt(-cv_mse_scores)

# Calculate the average CV-RMSE
cv_rmse.mean()

# Compare train and test RMSE
print("Train RMSE:", np.sqrt(mean_squared_error(y_train, reg_model.predict(X_train))), "Test RMSE:", np.sqrt(mean_squared_error(y_test, reg_model.predict(X_test))))

# Configure shuffled K-fold cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)


## 03. OLS (Simple, Multiple, Statsmodels, Manual)

In [ ]:
# ===== 03 | OLS (SIMPLE / MULTIPLE / STATSMODELS / MANUAL) =====

# Enter x and y values for a manual OLS exercise
x, y_values, new_x = np.array([40, 60, 80]), np.array([1200, 1600, 2000]), 95

# Calculate OLS slope manually from x and y arrays
slope = np.sum((x - np.mean(x)) * (y_values - np.mean(y_values))) / np.sum((x - np.mean(x)) ** 2)

# Calculate OLS intercept manually
intercept = np.mean(y_values) - slope * np.mean(x)

# Calculate a prediction from the simple OLS line
predicted_value = intercept + slope * new_x

# Check interpolation versus extrapolation
is_interpolation = np.min(x) <= new_x <= np.max(x)

# Fit simple OLS using a formula (intercept included)
ols_model = smf.ols("price ~ area", data=train_df).fit()

# Fit multiple OLS using a formula (intercept included)
ols_model = smf.ols("price ~ area + rooms", data=train_df).fit()

# [MOCK] Fit multiple OLS with explicit intercept (sm.OLS)
ols_model = sm.OLS(y_train, sm.add_constant(X_train)).fit()

# [MOCK] Display the model summary (coefficients, p-values, R²)
print(ols_model.summary())

# [MOCK] Display OLS coefficients including the intercept
ols_model.params

# Display coefficient p-values
ols_model.pvalues

# Display confidence intervals for coefficients
ols_model.conf_int(alpha=0.05)

# [MOCK] Calculate OLS training R-squared
ols_model.rsquared

# [MOCK] Convert R² to explained variability in percent
print(f"Explained variability: {ols_model.rsquared * 100:.1f}%")

# Get adjusted R-squared
ols_model.rsquared_adj

# Predict test data with statsmodels (add the intercept)
y_pred = ols_model.predict(sm.add_constant(X_test, has_constant="add"))

# Predict new data with a formula-based OLS model
ols_model.predict(pd.DataFrame({"area": [95], "rooms": [3.5]}))

# Predict new data with sm.OLS (same feature order + intercept)
ols_model.predict(sm.add_constant(new_X[X_train.columns], has_constant="add"))

# Calculate prediction intervals and mean confidence intervals
ols_model.get_prediction(pd.DataFrame({"area": [95], "rooms": [3.5]})).summary_frame(alpha=0.05)

# Get residuals from fitted OLS
residuals = ols_model.resid

# Plot residuals against fitted predictions
plt.scatter(ols_model.fittedvalues, ols_model.resid); plt.axhline(0, color="red"); plt.show()

# Show a Q-Q plot of residuals
sm.qqplot(ols_model.resid, line="45"); plt.show()

# Plot a histogram of residuals
plt.hist(ols_model.resid, bins=20); plt.show()

# Plot points and a fitted regression line (formula-based OLS only)
plt.scatter(train_df["area"], train_df["price"]); plt.plot(train_df["area"].sort_values(), ols_model.predict(train_df.sort_values("area")), color="red"); plt.show()

# Check correlation between numerical predictors
train_df[["area", "rooms"]].corr()

# Enter multiple OLS coefficients from a given output
beta0, beta_age, beta_mileage, beta_equipment, beta_ebike = 1318.0229, -99.9863, -0.0265, 215.7238, 1692.6052

# Enter the new observation for manual OLS prediction
age, mileage, equipment, is_ebike = 4, 6000, 3, 1

# [MOCK] Predict a multiple OLS value manually using coefficients
predicted_price = beta0 + beta_age * age + beta_mileage * mileage + beta_equipment * equipment + beta_ebike * is_ebike

## 04. Maximum Likelihood & Gradient Descent

In [ ]:
# ===== 04 - MAXIMUM LIKELIHOOD AND GRADIENT DESCENT =====

# Enter observed successes and failures for MLE
n_success, n_failure, p = 7, 3, 0.7

# Calculate Bernoulli MLE from successful and failed trials
p_hat = n_success / (n_success + n_failure)

# Calculate Bernoulli log-likelihood for a trial probability
log_likelihood = n_success * np.log(p) + n_failure * np.log1p(-p)

# Set the initial value, learning rate, and number of steps
b, learning_rate, n_steps = 0.0, 0.1, 10

# Evaluate the squared-error objective for a 1D parameter
loss = (b - 3) ** 2

# Calculate the gradient of the 1D loss
gradient = 2 * (b - 3)

# Perform one gradient descent update
b = b - learning_rate * gradient

# Perform multiple gradient descent updates (same 1D loss)
for _ in range(n_steps): b -= learning_rate * 2 * (b - 3)

## 05. Regression Trees, Splits, Pruning

In [ ]:
# 05 ===== REGRESSION TREES / SPLITS / PRUNING =====

# Enter example target values for regression tree leaves
parent_y, left_y, right_y = np.array([100, 200, 300, 400]), np.array([100, 200]), np.array([300, 400])

# Choose an example leaf to calculate its prediction
leaf_y = left_y

# Calculate a regression tree leaf prediction
leaf_prediction = np.mean(leaf_y)

# Calculate SSE for a regression leaf
leaf_sse = np.sum((leaf_y - np.mean(leaf_y)) ** 2)

# Calculate total SSE for a candidate split
split_sse = np.sum((left_y - np.mean(left_y)) ** 2) + np.sum((right_y - np.mean(right_y)) ** 2)

# Calculate SSE reduction of a candidate split
sse_reduction = np.sum((parent_y - np.mean(parent_y)) ** 2) - split_sse

# [MOCK] Create a regression tree
reg_model = DecisionTreeRegressor(max_depth=3, random_state=42)

# [MOCK] Train the regression tree
reg_model.fit(X_train, y_train)

# [MOCK] Predict regression values for the test data
y_pred = reg_model.predict(X_test)

# [MOCK] Calculate regression tree test R²
print("Tree test R²:", round(r2_score(y_test, y_pred), 3))

# [MOCK] Visualize the regression tree
plt.figure(figsize=(16, 6)); plot_tree(reg_model, feature_names=list(X_train.columns), filled=True, rounded=True); plt.show()

# Extract the regression tree rules as text
print(export_text(reg_model, feature_names=list(X_train.columns)))

# [MOCK] Identify the first-split feature of the tree
X_train.columns[reg_model.tree_.feature[0]]

# Show the number of tree leaves
reg_model.get_n_leaves()

# Show the depth of the fitted tree
reg_model.get_depth()

# Set tree complexity parameters
reg_model = DecisionTreeRegressor(max_depth=5, min_samples_split=4, min_samples_leaf=2, max_leaf_nodes=20, max_features=1.0, criterion="squared_error", random_state=42)

# Generate cost-complexity pruning candidates
pruning_path = reg_model.cost_complexity_pruning_path(X_train, y_train)

# Fit a pruned regression tree
pruned_reg = DecisionTreeRegressor(ccp_alpha=0.01, random_state=42).fit(X_train, y_train)

# Compare regression pruning results across alpha values
pruning_results = [(a, (m := DecisionTreeRegressor(ccp_alpha=a, random_state=42).fit(X_train, y_train)).get_n_leaves(), mean_squared_error(y_train, m.predict(X_train))) for a in [0.0, 0.001, 0.01]]

## 06. Bagging, Random Forest, Gradient Boosting (Regression)

In [ ]:
# 06 ===== BAGGING / RANDOM FOREST / GRADIENT BOOSTING (REGRESSION) =====

# Create a bagging regression model
reg_model = BaggingRegressor(estimator=DecisionTreeRegressor(), n_estimators=100, random_state=42)

# Train the bagging regressor
reg_model.fit(X_train, y_train)

# [MOCK] Create a random forest regressor (100 trees, depth 5)
reg_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

# [MOCK] Train the random forest regressor
reg_model.fit(X_train, y_train)

# [MOCK] Predict regression prices with the random forest
y_pred = reg_model.predict(X_test)

# [MOCK] Calculate random forest test R²
print("Random forest test R²:", round(r2_score(y_test, y_pred), 3))

# Create a random forest with out-of-bag (OOB) evaluation
reg_model = RandomForestRegressor(n_estimators=100, bootstrap=True, oob_score=True, random_state=42)

# Display the random forest OOB R² (after fitting)
reg_model.oob_score_

# Create a gradient boosting regressor
reg_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=2, random_state=42)

# Train the gradient boosting regressor
reg_model.fit(X_train, y_train)

# Compare multiple regression models by mean CV-RMSE
regression_scores = {name: np.sqrt(-cross_val_score(model, X_train, y_train, cv=5, scoring="neg_mean_squared_error")).mean() for name, model in {"OLS": LinearRegression(), "Tree": DecisionTreeRegressor(max_depth=3, random_state=42), "Forest": RandomForestRegressor(n_estimators=100, random_state=42), "Boosting": GradientBoostingRegressor(random_state=42)}.items()}

# [MOCK] Show impurity-based feature importances
feature_importance = pd.Series(reg_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

# [MOCK] Display sorted feature importance as a table
feature_importance.to_frame("importance")

# [MOCK] Plot impurity-based feature importances
feature_importance.sort_values().plot.barh(title="Feature importance"); plt.show()

# Calculate permutation importance on held-out test data
perm = permutation_importance(reg_model, X_test, y_test, n_repeats=10, random_state=42, scoring="r2")

# Display mean permutation importances
pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)

# [MOCK] Compare regression model test R² values
print("Tree R²:", r2_score(y_test, tree_model.predict(X_test)), "Forest R²:", r2_score(y_test, forest_model.predict(X_test)))

## 07. Logistic Regression, Probabilities

In [ ]:
# 07 ===== LOGISTIC REGRESSION / PROBABILITIES =====

# Calculate a logistic linear score
z = beta0 + beta1 * x1 + beta2 * x2

# Convert a logistic score into a probability
probability = 1 / (1 + np.exp(-z))

# Convert predicted probability to class with threshold 0.5
predicted_class = int(probability >= 0.5)

# Calculate the odds from a probability
odds = probability / (1 - probability)

# Create a logistic regression classifier
clf_model = LogisticRegression(max_iter=1000, random_state=42)

# Fit logistic regression on training data
clf_model.fit(X_train_class, y_train_class)

# Get logistic regression coefficients
pd.Series(clf_model.coef_[0], index=X_train_class.columns)

# Get logistic regression intercept
clf_model.intercept_[0]

# Predict positive-class probabilities
y_prob = clf_model.predict_proba(X_test_class)[:, 1]

# Predict test classes with the default threshold
y_pred_class = clf_model.predict(X_test_class)

# Apply a custom classification threshold
y_pred_class = (y_prob >= 0.30).astype(int)

# Compute predictions for a range of thresholds
predictions_by_threshold = {t: (y_prob >= t).astype(int) for t in [0.3, 0.5, 0.7]}



## 08. Confusion Matrix, Classification Metrics, Error Cost

In [ ]:
# 08 ===== CONFUSION MATRIX / CLASSIFICATION METRICS / ERROR COSTS =====

# [MOCK] Generate the confusion matrix (rows=actual, columns=predicted)
confusion_matrix(y_test_class, y_pred_class, labels=[0, 1])

# [MOCK] Extract TN, FP, FN and TP from the matrix
tn, fp, fn, tp = confusion_matrix(y_test_class, y_pred_class, labels=[0, 1]).ravel()

# Draw a confusion matrix with actual and predicted classes
ConfusionMatrixDisplay.from_predictions(y_test_class, y_pred_class, labels=[0, 1], display_labels=["0", "1"]); plt.show()

# [MOCK] Enter TN, FP, FN, TP manually if given in the exam
tn, fp, fn, tp = 175, 6, 28, 31

# Calculate accuracy manually
accuracy = (tp + tn) / (tp + tn + fp + fn)

# Calculate precision manually
precision = tp / (tp + fp)

# [MOCK] Calculate recall (sensitivity) manually
recall = tp / (tp + fn)

# Calculate specificity (true negative rate)
specificity = tn / (tn + fp)

# Calculate false positive rate
fpr = fp / (fp + tn)

# Calculate F1-score manually
f1 = 2 * precision * recall / (precision + recall)

# [MOCK] Calculate accuracy with scikit-learn
accuracy_score(y_test_class, y_pred_class)

# Calculate precision with scikit-learn
precision_score(y_test_class, y_pred_class, zero_division=0)

# [MOCK] Calculate recall with scikit-learn
recall_score(y_test_class, y_pred_class, zero_division=0)

# Calculate F1-score with scikit-learn
f1_score(y_test_class, y_pred_class, zero_division=0)

# Show all main classification metrics
print(classification_report(y_test_class, y_pred_class, zero_division=0))

# [MOCK] Calculate majority-class baseline accuracy
baseline_accuracy = y_test_class.value_counts(normalize=True).max()

# Enter the costs of a false positive and false negative
cost_fp, cost_fn = 10, 100

# Calculate the total classification error cost
total_cost = fp * cost_fp + fn * cost_fn

# Evaluate a PREDEFINED threshold on test data (never choose threshold using test results)
tn, fp, fn, tp = confusion_matrix(y_test_class, (y_prob >= 0.30).astype(int), labels=[0, 1]).ravel()

# Split the existing training data for threshold selection (keep the test set untouched)
X_subtrain, X_val, y_subtrain, y_val = train_test_split(X_train_class, y_train_class, test_size=0.25, random_state=42, stratify=y_train_class)

# Fit a temporary classifier on subtraining data only
validation_model = LogisticRegression(max_iter=1000, random_state=42).fit(X_subtrain, y_subtrain)

# Predict positive-class probabilities on validation data
val_probs = validation_model.predict_proba(X_val)[:, 1]

# Compare error costs for candidate thresholds using VALIDATION data
cost_by_threshold = {t: (lambda c: c[0, 1] * cost_fp + c[1, 0] * cost_fn)(confusion_matrix(y_val, (val_probs >= t).astype(int), labels=[0, 1])) for t in [0.3, 0.5, 0.7]}

# Select the lowest-cost validation threshold (not from the test set)
best_threshold = min(cost_by_threshold, key=cost_by_threshold.get)

# Refit the final classifier on the entire training split after threshold selection
final_model = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_class, y_train_class)

# Evaluate the selected threshold ONCE on untouched test data
print(confusion_matrix(y_test_class, (final_model.predict_proba(X_test_class)[:, 1] >= best_threshold).astype(int), labels=[0, 1]))


## 09. ROC, AUC, Classification Tree

In [ ]:
# 09 ===== ROC / AUC / CLASSIFICATION TREE =====

# [MOCK] Calculate ROC AUC from positive-class probabilities
auc = roc_auc_score(y_test_class, y_prob)

# [MOCK] Draw ROC curve from a trained estimator
RocCurveDisplay.from_estimator(clf_model, X_test_class, y_test_class); plt.show()

# Draw ROC curve from actual classes and probabilities
RocCurveDisplay.from_predictions(y_test_class, y_prob); plt.show()

# Show the ROC diagonal reference line
plt.plot([0, 1], [0, 1], "--", color="grey")

# [MOCK] Create a classification tree
clf_model = DecisionTreeClassifier(max_depth=3, random_state=42)

# [MOCK] Train the classification tree
clf_model.fit(X_train_class, y_train_class)

# Predict test classes with a tree
y_pred_class = clf_model.predict(X_test_class)

# [MOCK] Predict positive-class probabilities with a tree
y_prob = clf_model.predict_proba(X_test_class)[:, 1]

# Draw a classification tree
plt.figure(figsize=(15, 6)); plot_tree(clf_model, feature_names=list(X_train_class.columns), class_names=["0", "1"], filled=True); plt.show()

# Display the classification tree as rules
print(export_text(clf_model, feature_names=list(X_train_class.columns)))

# Identify the first split feature of the classification tree
X_train_class.columns[clf_model.tree_.feature[0]]

# Get the order of predicted classes
clf_model.classes_

# Enter the positive and negative counts for a Gini exercise
n_pos, n_neg = 3, 7

# Calculate binary Gini impurity from positive/negative counts
gini = 1 - (n_pos / (n_pos + n_neg)) ** 2 - (n_neg / (n_pos + n_neg)) ** 2

# Enter child sizes and impurities for a candidate split
n_left, n_right, n_total, gini_left, gini_right, gini_parent = 5, 5, 10, 0.32, 0.48, 0.42

# Calculate weighted Gini impurity for a split
weighted_gini = (n_left / n_total) * gini_left + (n_right / n_total) * gini_right

# Calculate Gini decrease from a candidate split
gini_gain = gini_parent - weighted_gini

# Compute a leaf positive-class probability
leaf_probability = n_pos / (n_pos + n_neg)

# Predict a leaf majority class
leaf_class = int(leaf_probability >= 0.5)

# Generate candidate tree pruning alpha values
pruning_path = clf_model.cost_complexity_pruning_path(X_train_class, y_train_class)

# Fit a pruned classification tree
pruned_clf = DecisionTreeClassifier(ccp_alpha=0.01, random_state=42).fit(X_train_class, y_train_class)

# Compare classification pruning results across alpha values
classification_pruning = [(a, (m := DecisionTreeClassifier(ccp_alpha=a, random_state=42).fit(X_train_class, y_train_class)).get_n_leaves(), m.score(X_train_class, y_train_class)) for a in [0.0, 0.01]]


## 10. Random Forest, Gradient Boosting (Classification)

In [ ]:
# 10 ===== RANDOM FOREST / GRADIENT BOOSTING (CLASSIFICATION) =====

# [MOCK] Create a random forest classifier with 100 trees
clf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Create a random forest with feature subsampling
clf_model = RandomForestClassifier(n_estimators=100, max_features="sqrt", random_state=42)

# [MOCK] Train the random forest classifier
clf_model.fit(X_train_class, y_train_class)

# [MOCK] Predict test classes (not training classes!)
y_pred_class = clf_model.predict(X_test_class)

# Predict positive-class probabilities
y_prob = clf_model.predict_proba(X_test_class)[:, 1]

# [MOCK] Evaluate test accuracy (not training accuracy!)
print("Test accuracy:", round(accuracy_score(y_test_class, y_pred_class), 3))

# [MOCK] Evaluate a random forest confusion matrix on test data
print(confusion_matrix(y_test_class, y_pred_class, labels=[0, 1]))

# Create a gradient boosting classifier
clf_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=2, random_state=42)

# Train gradient boosting classifier
clf_model.fit(X_train_class, y_train_class)

# Evaluate classification ROC AUC on the test set
roc_auc_score(y_test_class, clf_model.predict_proba(X_test_class)[:, 1])

# Show classification random forest feature importances
pd.Series(clf_model.feature_importances_, index=X_train_class.columns).sort_values(ascending=False)


## 11. Classification CV, Grid Search, Model Selection

In [ ]:
# 11 ===== CLASSIFICATION CV / GRID SEARCH / MODEL SELECTION =====

# Compute 5-fold cross-validation AUC on training data
cv_auc = cross_val_score(clf_model, X_train_class, y_train_class, cv=5, scoring="roc_auc")

# Calculate mean cross-validation AUC
cv_auc.mean()

# Create shuffled stratified cross-validation folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Compare classifier models using mean CV-AUC
classification_scores = {name: cross_val_score(model, X_train_class, y_train_class, cv=5, scoring="roc_auc").mean() for name, model in {"Logistic": LogisticRegression(max_iter=1000), "Tree": DecisionTreeClassifier(max_depth=3, random_state=42), "Forest": RandomForestClassifier(n_estimators=100, random_state=42), "Boosting": GradientBoostingClassifier(random_state=42)}.items()}

# Define hyperparameters to evaluate in a grid search
param_grid = {"max_features": ["sqrt", 1.0], "min_samples_leaf": [1, 5, 20]}

# Create GridSearchCV using classification AUC
grid = GridSearchCV(RandomForestClassifier(n_estimators=100, random_state=42), param_grid=param_grid, cv=5, scoring="roc_auc", refit=True)

# Fit GridSearchCV using training data only
grid.fit(X_train_class, y_train_class)

# Get the best hyperparameter settings
grid.best_params_

# Get the best cross-validation AUC
grid.best_score_

# Get the number of tested combinations
len(grid.cv_results_["params"])

# Use the refitted best model for test predictions
best_model = grid.best_estimator_

# Evaluate the tuned model on test data
roc_auc_score(y_test_class, grid.predict_proba(X_test_class)[:, 1])

# Inspect the ranked grid search results
pd.DataFrame(grid.cv_results_).sort_values("rank_test_score")[["params", "mean_test_score", "rank_test_score"]].head()


## 12. Exam Error-Analysis Checks

In [ ]:
# 12 ===== EXAM ERROR-ANALYSIS CHECKS =====

# [MOCK] Check test_size = 0.20 for an 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# [MOCK] Check no target leakage in regression features
assert "price_chf" not in X.columns

# [MOCK] Check the intercept in sm.OLS (add_constant)
ols_model = sm.OLS(y_train, sm.add_constant(X_train)).fit()

# [MOCK] Check regression R² on the correct dataset
print("Train R²:", ols_model.rsquared, "Test R²:", r2_score(y_test, ols_model.predict(sm.add_constant(X_test, has_constant="add"))))

# [MOCK] Check that classification predictions use X_test_class
y_pred_class = clf_model.predict(X_test_class)

# [MOCK] Check that classifier evaluation uses y_test_class
print(accuracy_score(y_test_class, y_pred_class), confusion_matrix(y_test_class, y_pred_class))

# [MOCK] Check stratification with class distribution
print(y_train_class.value_counts(normalize=True), y_test_class.value_counts(normalize=True))

# [MOCK] Calculate recall for a supplied confusion matrix
recall = tp / (tp + fn)

# [MOCK] Check the positive-class probability for AUC
roc_auc_score(y_test_class, clf_model.predict_proba(X_test_class)[:, 1])

## Prüfungsinterpretation in Kürze

| Thema | Prüfungsreife Interpretation |
|---|---|
| Supervised Learning | Regression: numerisches `y` (Preis/Verbrauch); Klassifikation: Klassen `y` (Service 0/1). Features = X, Zielvariable = y. |
| MSE / RMSE | Je kleiner, desto besser. MSE quadriert die Fehler; RMSE hat dieselbe Einheit wie die Zielvariable. |
| R² / Adjusted R² | R² beschreibt den auf den betrachteten Daten erklärten Variabilitätsanteil; `0.966` = 96.6 %. Test-R² kann negativ sein. Adjusted R² berücksichtigt die Anzahl Features. |
| Koeffizient in multipler OLS | Veränderung des erwarteten Zielwerts pro +1 beim betreffenden Merkmal, **bei konstanten übrigen Merkmalen** (*ceteris paribus*). 0/1-Feature: Unterschied zwischen beiden Gruppen bei sonst gleichen Merkmalen. Kein automatischer Kausalbeweis. |
| Konfidenz-/Vorhersageintervall | `mean_ci_*`: Unsicherheit der **mittleren** Zielvorhersage; `obs_ci_*`: Bereich für **eine einzelne** neue Beobachtung, typischerweise breiter. |
| Residuenanalyse | Systematische Muster oder Trichterform: mögliche Modellprobleme; Q-Q-Plot: grobe Normalitätsbeurteilung der Residuen. |
| Loss vs. objective vs. metric | *Loss* bewertet eine Vorhersage, das *Trainingsziel* minimiert z. B. durchschnittlichen Loss (ggf. mit Strafterm), eine *Evaluationsmetrik* beurteilt die Modellqualität – nicht zwingend identisch mit dem Trainingsziel. |
| Bias–Variance Trade-off | Hoher Bias = zu starre Modellannahmen (häufig Underfitting); hohe Varianz = starke Abhängigkeit von zufälligen Trainingsdetails (häufig Overfitting). |
| Under-/Overfitting | Underfitting: Train und Test schlecht; Overfitting: Train sehr gut, Test deutlich schlechter. CV/Pruning/Komplexitätsbegrenzung können helfen. |
| CV | Nur Trainingsdaten in Folds aufteilen. Negative CV-MSE-Werte sind sklearn-Konvention: für RMSE zuerst Vorzeichen wechseln, dann Wurzel. Testset erst am Schluss. |
| Regression Tree / Random Forest | Regression Tree sagt den Mittelwert pro Blatt vorher, wählt Splits nach kleiner SSE; Random Forest mittelt viele Bäume, kann Varianz reduzieren. Mehr Test-R² ist besser. |
| Feature Importance | Zeigt Nutzung/Wichtigkeit fürs Modell, aber **keinen kausalen Effekt**. Impurity Importance kann verzerrt sein; Permutation Importance misst Score-Verschlechterung beim Mischen. |
| Bagging vs. Boosting | Bagging: parallel/auf Bootstrap-Stichproben, mittelt Vorhersagen; Boosting: nacheinander, korrigiert bisherige Fehler. |
| Logistic Regression | Score `z` via Sigmoid zu Wahrscheinlichkeit; Klasse 1 ab Schwelle `τ` (oft 0.5). |
| Confusion Matrix | In sklearn bei `labels=[0,1]`: `[[TN, FP], [FN, TP]]`; TP = richtig positiv, TN = richtig negativ, FP = Fehlalarm, FN = übersehener positiver Fall. |
| Accuracy | `(TP+TN)/N`; kann bei unausgeglichenen Klassen irreführend sein. Majority-Class-Baseline beachten. |
| Precision / Recall / F1 | Precision = TP/(TP+FP): Qualität positiver Alarme. Recall = TP/(TP+FN): Anteil erkannter tatsächlicher Positiver. F1 = harmonisches Mittel. |
| Fehlerkosten / Threshold | Niedrigere Schwelle → meist mehr TP und FP, weniger FN. Bei hohen FN-Kosten Recall besonders relevant; Schwelle auf Validierungsdaten nach Kosten/Ziel wählen, **nicht auf dem Testset**. |
| ROC/AUC | ROC: TPR (Recall) gegen FPR über alle Schwellen. AUC 0.5 ≈ zufällige Rangtrennung; 1.0 = perfekte Rangtrennung. Für AUC Wahrscheinlichkeiten/Scores statt harter Klassen verwenden. |
| Gini / Klassifikationsbaum | Gini 0 = reiner Knoten. Bester Split reduziert gewichtete Unreinheit; Klassenwahrscheinlichkeit im Blatt = Positiv-Anteil. |
| MLE / Gradient Descent | MLE wählt Parameter mit höchster Likelihood der Daten; Gradient Descent verändert Parameter schrittweise entlang des negativen Gradienten, Lernrate bestimmt Schrittgrösse. |
| SVM | Konzept aus Spick 2: maximale Margin, Support Vectors bestimmen Grenze; `C` regelt Strafgewicht für Margin-Verletzungen, Kernel ermöglicht nichtlineare Trennflächen. Skalierung auf Training lernen, in CV über Pipeline. **Nur Theorie im Original, keine Code-Übung.** |


## Mock Exam – obligatorische Fehleranalyse-Checkliste

**B1 (OLS):** Zielvariable ist **nicht** gleichzeitig ein Feature (`price_chf` aus X entfernen → sonst Target Leakage und künstliches R²=1). `sm.OLS()` hat **ohne `sm.add_constant()` keinen Intercept**. Aus dem korrigierten Trainingsmodell: R² ≈ **0.966 = 96.6 %** (nur das gegebene Mock Exam).

**B2 (Random-Forest-Klassifikation):** `test_size=0.80` bedeutet 80 % Test statt 20 % → `test_size=0.20`. `predict(X_train)` und `accuracy_score(y_train, ...)` prüfen bekannte Daten, nicht Generalisierung → konsequent `X_test_class` und `y_test_class`. Im Mock Exam: Test-Accuracy ≈ **0.887**, Recall für Servicebedarf **43/59 ≈ 0.73**. Zahlen nicht auf andere Daten übertragen.

**A1:** Preis (`price_chf`) = Regression, Test-R² oder Test-RMSE; Service (`service_needed`) = Klassifikation, etwa Recall, Precision, Accuracy, F1, ROC-AUC.

**A2:** OLS-Koeffizient `is_ebike` ≈ +CHF 1'693 **bei gleichem Alter, Kilometerstand und gleicher Ausstattung**. Preis per Hand: `β0 + β1*age_years + β2*mileage_km + β3*equipment_level + β4*is_ebike`.

**A3:** Matrix `[[175,6],[28,31]]` → TN=175, FP=6, FN=28, TP=31 → Recall = `31/(31+28) ≈ 0.53`. Ohne Service-Bedarf vorherzusagen hätte schon `181/240 ≈ 75 %` Accuracy; bei sicherheitsrelevantem Service sind **FN** besonders kritisch.

**C1:** `DecisionTreeRegressor(max_depth=3, random_state=42)`, `.fit(X_train,y_train)`, `.predict(X_test)`, `r2_score(y_test, ...)`. Erster Split im Mock Exam: `is_ebike` (weil das die SSE stark reduziert), R² ≈ 0.911.

**C2:** `RandomForestRegressor(n_estimators=100,max_depth=5,random_state=42)` plus Test-R² und `.feature_importances_`. Im Mock Exam R² ≈ 0.957; `is_ebike` am wichtigsten. **0.957 > 0.911**.

**C3:** `DecisionTreeClassifier(max_depth=3,random_state=42)` mit `X_train_class,y_train_class`; ROC/AUC auf `X_test_class,y_test_class`. Im Mock Exam AUC ≈ 0.90 (gute Trennung).

**Prüfe bei jedem Codefehler:** ① Richtige Zielvariable/Features, keine Leakage? ② Train/Test und Splitprozente korrekt? ③ Intercept bei `sm.OLS`? ④ Regression vs Klassifikation/geeignetes Modell? ⑤ `.fit()` nur auf Train? ⑥ `.predict()` und Metriken auf **Test**? ⑦ `predict_proba[:,1]` für ROC/AUC? ⑧ Parameter exakt wie verlangt? ⑨ Kennzahl korrekt interpretiert – vor allem Accuracy bei seltenen positiven Fällen?

**Threshold-Tuning:** Die Testdaten dürfen nicht genutzt werden, um unter mehreren Schwellenwerten den besten auszuwählen; dafür eine interne Validierung verwenden.
